<a href="https://colab.research.google.com/github/Aijamal24/ML_Practice/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('o_churn_synthetic.csv')
X = df.drop (columns=['churn', 'subscriber_code'])
y = df[ 'churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

def make_prep():
  return ColumnTransformer (
      [('cat', OneHotEncoder (handle_unknown='ignore'), ['region', 'tariff'])],
      remainder='passthrough')
print ('train:', X_train.shape, 'test:', X_test.shape)

train: (2400, 11) test: (600, 11)


In [7]:
import time
from sklearn.metrics import f1_score, roc_auc_score

def fit_and_report(name, pipe):
  t0 = time.perf_counter()
  pipe.fit(X_train, y_train)
  sec = time.perf_counter() - t0
  f1_tr = f1_score(y_train, pipe.predict(X_train))
  f1_te = f1_score(y_test, pipe.predict(X_test))
  auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
  print (f"{name:<14} F1 train={f1_tr:.3f} F1 test={f1_te:.3f}"
         f"AUC test={auc:.3f} {sec:.2f} c")

In [8]:
from xgboost import XGBClassifier

xgb = make_pipeline(make_prep(), XGBClassifier(
    n_estinators=300, Learning_rate=0.05, max_depth=4, random_state=42))
fit_and_report("XBoost", xgb)

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [03:48:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "Learning_rate", "n_estinators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XBoost         F1 train=0.904 F1 test=0.564AUC test=0.761 0.31 c


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(make_prep(), RandomForestClassifier (
    n_estimators=100, random_state=42))
fit_and_report ("Случайный лес", rf)

Случайный лес  F1 train=1.000 F1 test=0.556AUC test=0.752 1.64 c


In [14]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split( X_train, y_train, test_size=0.2, stratify=y_train, random_state=42 )
p = make_prep().fit(X_tr)
Xtr, Xval = p.transform(X_tr), p.transform(X_val)
print("строк:", len(X_tr), len(X_val), len(X_test),
      " признаков после one-hot:", Xtr.shape[1])

строк: 1920 480 600  признаков после one-hot: 22


In [15]:
es = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4,
                   eval_metric="logloss", early_stopping_rounds=50,
                   random_state=42)
es.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
print("лучшая итерация:", es.best_iteration,
      "\nlogloss на val:", round(es.best_score, 4))

лучшая итерация: 78 
logloss на val: 0.5405


In [17]:
for lr in [0.3, 0.1, 0.05, 0.01]:
    for depth in [3, 4, 6]:
        m = XGBClassifier(n_estimators=3000, learning_rate=lr,
                          max_depth=depth, eval_metric="logloss",
                          early_stopping_rounds=50, random_state=42)
        m.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
        auc = roc_auc_score(y_val, m.predict_proba(Xval)[:, 1])
        n = m.best_iteration + 1
        print(f"lr={lr:<5} depth={depth} деревьев={n:<4} AUC val={auc:.3f}")

lr=0.3   depth=3 деревьев=19   AUC val=0.770
lr=0.3   depth=4 деревьев=11   AUC val=0.773
lr=0.3   depth=6 деревьев=8    AUC val=0.757
lr=0.1   depth=3 деревьев=72   AUC val=0.778
lr=0.1   depth=4 деревьев=47   AUC val=0.780
lr=0.1   depth=6 деревьев=36   AUC val=0.766
lr=0.05  depth=3 деревьев=127  AUC val=0.783
lr=0.05  depth=4 деревьев=79   AUC val=0.776
lr=0.05  depth=6 деревьев=85   AUC val=0.768
lr=0.01  depth=3 деревьев=652  AUC val=0.780
lr=0.01  depth=4 деревьев=511  AUC val=0.775
lr=0.01  depth=6 деревьев=307  AUC val=0.763


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaled = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
    remainder=StandardScaler()
)

models = {
    "Логрегрессия": make_pipeline(scaled, LogisticRegression(max_iter=1000)),
    "Случайный лес": make_pipeline(make_prep(), RandomForestClassifier(
        n_estimators=100, random_state=42)),
    "XGBoost": make_pipeline(make_prep(), XGBClassifier(
        n_estimators=127, learning_rate=0.05, max_depth=3, random_state=42)),
}

for name, pipe in models.items():
    fit_and_report(name, pipe)

Логрегрессия   F1 train=0.594 F1 test=0.611AUC test=0.806 0.04 c
Случайный лес  F1 train=1.000 F1 test=0.556AUC test=0.752 0.86 c
XGBoost        F1 train=0.645 F1 test=0.586AUC test=0.792 0.52 c
